# 02 – Data Preprocessing & Feature Engineering
## FraudShield AI · PaySim

This notebook walks through every preprocessing and feature-engineering step performed
by `src/data_preprocessing.py` and `src/feature_engineering.py`.

**Steps covered:**
1. Raw data loading and schema review
2. Data cleaning (duplicates, outliers)
3. Filtering fraud-susceptible transaction types
4. Stratified train / test split
5. Feature engineering (domain + ratio features)
6. One-hot encoding and StandardScaler normalization
7. Final feature matrix inspection


In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from src.data_preprocessing   import load_raw_data, clean_data, filter_fraud_susceptible, split_data_stratified
from src.feature_engineering  import create_features, preprocess_features

print("Imports OK ✓")


## 2.1 Load Raw Data

In [ ]:
RAW_PATH = "data/raw/PS_20174392719_1491204439457_log.csv"
df_raw = load_raw_data(RAW_PATH)
print(f"Loaded  {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.dtypes


## 2.2 Data Cleaning

In [ ]:
df_clean = clean_data(df_raw)
print(f"After cleaning: {df_clean.shape[0]:,} rows")

# Verify no missing values remain
missing_after = df_clean.isna().sum().sum()
dup_after     = df_clean.duplicated().sum()
print(f"Missing values : {missing_after}")
print(f"Duplicates     : {dup_after}")


## 2.3 Filter Fraud-Susceptible Transactions

In [ ]:
df_filtered = filter_fraud_susceptible(df_clean)
print(f"\nAll transactions  : {len(df_clean):,}")
print(f"TRANSFER+CASH_OUT : {len(df_filtered):,}  ({len(df_filtered)/len(df_clean)*100:.1f}%)")
print(f"Fraud in filtered : {df_filtered['isFraud'].sum():,}  ({df_filtered['isFraud'].mean()*100:.3f}%)")

print("\nType breakdown:")
print(df_filtered['type'].value_counts())


## 2.4 Stratified Train / Test Split (80 / 20)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = split_data_stratified(df_filtered)

print(f"Train size : {len(X_train_raw):,}  |  Fraud: {y_train.sum():,}  ({y_train.mean()*100:.3f}%)")
print(f"Test  size : {len(X_test_raw):,}  |  Fraud: {y_test.sum():,}  ({y_test.mean()*100:.3f}%)")

# Confirm stratification preserved fraud ratio
assert abs(y_train.mean() - y_test.mean()) < 0.002, "Stratification failed!"
print("\nStratification verified ✓ — fraud ratio preserved in both splits")


## 2.5 Feature Engineering

In [ ]:
# Apply on a sample for notebook speed
sample_df = df_filtered.sample(n=5000, random_state=42).copy()
sample_feat = create_features(sample_df)

new_cols = ['balance_difference_orig', 'balance_difference_dest',
            'transaction_amount_ratio', 'dest_is_merchant', 'transaction_velocity']

print("New features created:")
display(sample_feat[new_cols].describe().round(4))


In [ ]:
# Visualise the most important engineered feature
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes,
    ['balance_difference_orig', 'transaction_amount_ratio', 'balance_difference_dest'],
    ['Balance Diff Origin', 'Amount Ratio', 'Balance Diff Dest']):
    for label, color, mask in [
        ('Legit', '#3b82f6', sample_feat['isFraud']==0),
        ('Fraud', '#ef4444', sample_feat['isFraud']==1),
    ]:
        ax.hist(sample_feat.loc[mask, col].clip(
                    sample_feat[col].quantile(0.01),
                    sample_feat[col].quantile(0.99)),
                bins=50, alpha=0.65, color=color, label=label, edgecolor='none')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Engineered Feature Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("\nKey observation: balance_difference_orig ≈ 0 is the strongest fraud signal")


## 2.6 One-Hot Encoding & Standard Scaling

In [ ]:
# Build a small train/test pair from the sample for illustration
sample_train = sample_df.sample(frac=0.8, random_state=42).copy()
sample_train['isFraud'] = sample_df.loc[sample_train.index, 'isFraud']
sample_test  = sample_df.drop(sample_train.index).copy()
sample_test['isFraud'] = sample_df.loc[sample_test.index, 'isFraud']

SCALER_PATH = "models/scaler_demo.pkl"
X_demo_train = preprocess_features(sample_train, is_training=True,  scaler_path=SCALER_PATH)
X_demo_test  = preprocess_features(sample_test,  is_training=False, scaler_path=SCALER_PATH)

print(f"Feature matrix shape (train): {X_demo_train.shape}")
print(f"Feature matrix shape (test) : {X_demo_test.shape}")
print(f"\nFeature columns ({len(X_demo_train.columns)}):")
for c in X_demo_train.columns:
    print(f"  • {c}")


In [ ]:
# Confirm scaling
print("Post-scaling descriptive stats (train):")
display(X_demo_train.describe().round(3))


## 2.7 Feature Matrix Summary

In [ ]:
print("Final feature set used for modelling:")
feature_meta = {
    'step':                     'Time step (0–743 hours)',
    'amount':                   'Transaction amount ($)',
    'oldbalanceOrg':            "Sender's opening balance",
    'newbalanceOrig':           "Sender's closing balance",
    'oldbalanceDest':           "Receiver's opening balance",
    'newbalanceDest':           "Receiver's closing balance",
    'balance_difference_orig':  'oldBal − newBal − amount (fraud signal #1)',
    'balance_difference_dest':  'newBal_dest − oldBal_dest − amount',
    'transaction_amount_ratio': 'amount / oldBal (≈1 in fraud)',
    'dest_is_merchant':         '1 if receiver starts with M',
    'transaction_velocity':     '# txns by same sender in same step',
    'type_TRANSFER':            'One-hot: TRANSFER = 1',
    'type_CASH_OUT':            'One-hot: CASH_OUT = 1',
}
for feat, desc in feature_meta.items():
    print(f"  {feat:<32} {desc}")
